In [ ]:
# Is "queries are degraded, gallery scans are clean" actually TRUE?
#
# Every restoration idea rests on that premise and nobody has measured it. If the
# two populations do not separate on sharpness/resolution, then we cannot tell
# them apart, cannot restore selectively, and the whole direction dies here for
# 5 minutes of CPU instead of 2 hours of GPU plus a submission slot.
#
# This runs on CPU, so it can share a session with a GPU job or run in its own.
import glob, os, time
import numpy as np, cv2
from pathlib import Path
from multiprocessing import Pool

DATA = Path("/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset")
if not DATA.exists():
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if glob.glob(os.path.join(d, "*.png"))]
    assert hits, "no PNG directory under /kaggle/input"
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, "*.png")))))
paths = sorted(DATA.glob("*.png"))
print(len(paths), "images from", DATA)
assert len(paths) == 20000

def stats(p):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: return (0, 0, 0.0, 0.0, 0.0)
    h, w = im.shape
    s = cv2.resize(im, (512, 512), interpolation=cv2.INTER_AREA)
    lap = cv2.Laplacian(s, cv2.CV_64F).var()            # sharpness
    # high-frequency energy share: blurred images lose the outer FFT ring
    F = np.abs(np.fft.fftshift(np.fft.fft2(s.astype(np.float32))))
    cy = cx = 256
    yy, xx = np.ogrid[:512, :512]
    r = np.sqrt((yy-cy)**2 + (xx-cx)**2)
    hf = float(F[r > 128].sum() / (F.sum() + 1e-9))
    return (h, w, float(lap), hf, float(s.std()))

In [ ]:
t0 = time.time()
with Pool(os.cpu_count()) as pool:
    rows = pool.map(stats, paths, chunksize=64)
S = np.array(rows, dtype=np.float64)
np.save("/kaggle/working/img_stats.npy", S)
np.save("/kaggle/working/img_names.npy", np.array([p.name for p in paths]))
print(f"done in {(time.time()-t0)/60:.1f} min")

H, W, LAP, HF, STD = S[:,0], S[:,1], S[:,2], S[:,3], S[:,4]
px = H * W
for nm, v in [("min side", np.minimum(H,W)), ("megapixels", px/1e6),
              ("laplacian var", LAP), ("high-freq share", HF)]:
    q = np.percentile(v, [1,10,25,50,75,90,99])
    print(f"{nm:>17}: " + "  ".join(f"p{p}={x:,.4g}" for p,x in zip([1,10,25,50,75,90,99], q)))

In [ ]:
# The decisive plot, as numbers: is the corpus BIMODAL on sharpness?
# 1,000 queries + 10,000 gallery + 9,000 distractors. Gallery scans should form a
# sharp, high-resolution mode; visitor photos a soft, low-resolution one. If the
# histogram is unimodal, degradation is not separable and selective restoration
# is not implementable -- which is the answer, not a setback.
def hist(v, name, bins=12):
    lo, hi = np.percentile(v, [0.5, 99.5])
    h, e = np.histogram(np.clip(v, lo, hi), bins=bins)
    print(f"\n{name}")
    for c, l, r in zip(h, e[:-1], e[1:]):
        print(f"  {l:10.4g}..{r:<10.4g} {'#' * int(60*c/h.max()):<60} {c}")
hist(np.log10(LAP + 1e-6), "log10 laplacian variance")
hist(HF, "high-frequency energy share")
hist(np.log10(np.minimum(H, W)), "log10 min side (native resolution)")

In [ ]:
# Cross-check against the suspected query set. rot_candidates.npy holds the 3,439
# images an earlier orientation test flagged; whatever else they are, they should
# be enriched for real queries. If candidates are systematically softer than the
# rest, degradation detection works and restoration is implementable.
c = sorted(glob.glob("/kaggle/input/**/rot_candidates.npy", recursive=True))
if not c:
    print("attach rot_candidates.npy to run this check")
else:
    cand = np.load(c[0]); m = np.zeros(len(paths), bool); m[cand] = True
    print(f"candidates {m.sum()}  rest {(~m).sum()}\n")
    print(f"{'metric':>18}{'candidates':>14}{'rest':>12}{'ratio':>9}")
    for nm, v in [("laplacian var", LAP), ("high-freq share", HF),
                  ("min side", np.minimum(H,W)), ("std", STD)]:
        a, b = np.median(v[m]), np.median(v[~m])
        print(f"{nm:>18}{a:>14.4g}{b:>12.4g}{a/(b+1e-9):>9.2f}")
    # separability: how well does a single threshold on sharpness recover them?
    from numpy import argsort
    order = argsort(LAP)                       # softest first
    top = np.zeros(len(paths), bool); top[order[:m.sum()]] = True
    print(f"\nif we called the {m.sum()} SOFTEST images 'queries', "
          f"{(top & m).sum()} would be candidates "
          f"({(top & m).sum()/m.sum()*100:.1f}% overlap; {m.sum()/len(paths)*100:.1f}% expected by chance)")